# C2.5 - Simplifier student: LoRA on Colab + GGUF export (FinSight, M6)

Fine-tunes a ~4B Qwen instruct model to rewrite risk factors in plain English, from the filtered
teacher pairs (`train.jsonl`, `dev.jsonl` in `MyDrive/FinSight/student/`, made by
`python -m finsight.risks.simplify split`; split by company). The training rows already use the
serving prompt (`finsight.risks.simplify`).

- **PRECISION `bf16`** (L4/A100): LoRA r=16 on a bf16 base, no 4-bit. **`nf4`**: the T4 recipe (4-bit, fp16).
- Checkpoints go to local disk and are copied to Drive on every save (`SAVE_STEPS`); a disconnect resumes.
- Then the adapter is merged **temporarily** to make a **GGUF Q4_K_M** with llama.cpp. The merged fp16
  weights are deleted. Only the adapter and the GGUF are kept (Drive) and, with `UPLOAD`, pushed to a
  **private** Hugging Face repo. Never merged fp16.
- The teacher rewrites are AI-made, and the corpus terms are non-commercial: keep the repo private.
- Smoke first (`SMOKE = True`, T4, `nf4`), then the real run on an L4 or A100.

In [ ]:
# ---- parameters (edit, then Run all) ------------------------------------------------------
JOB = "student"                    # Drive folder MyDrive/FinSight/student/ holds train.jsonl and dev.jsonl
MODEL = "Qwen/Qwen3-4B-Instruct-2507"   # Apache-2.0, non-thinking; the zero-shot bake-off may change it
PRECISION = "bf16"                 # "bf16": LoRA on a bf16 base (L4/A100); "nf4": 4-bit QLoRA, the T4 recipe
EPOCHS = 2
LEARNING_RATE = 2e-4
BATCH_SIZE = 2                     # raise to 8 on an A100 in bf16
GRAD_ACCUM = 8
MAX_SEQ = 1024
SAVE_STEPS = 200
SEED = 2026
SMOKE = True                       # True: 64 train / 16 dev rows, 10 steps (run on a T4 with PRECISION = "nf4")
EXPORT_GGUF = True
UPLOAD = False                     # True: push adapter + GGUF (never merged fp16) to a private HF repo
HF_REPO = "AkshatTm/finsight-student-4b"
UNITS_BEFORE = None
UNITS_AFTER = None
UNASSIGN = False

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft>=0.13", "bitsandbytes>=0.45",
                "huggingface_hub"], check=True)

In [ ]:
# Shared helpers: copy notebooks/colab/_common.py to MyDrive/FinSight/_common.py once (COLAB_STEPS).
import sys, time
sys.path.insert(0, "/content/drive/MyDrive/FinSight")
from google.colab import drive
drive.mount("/content/drive")
from _common import mount_drive, Checkpointer, run_summary, gpu_info, unassign_runtime
STARTED = time.time()
print(gpu_info())

In [ ]:
# GENERATED from finsight.risks.simplify by scripts/make_student_notebook.py; do not edit
PROMPT_VERSION = 'student-v1'
INSTRUCTIONS = 'Rewrite this risk factor from an Indian IPO offer document in plain English for a first-time investor.\nRules: at most 60 words; sentences under 20 words; keep every number exactly as written, with its unit; keep how certain it is ("may" stays "may", "has" stays "has"); add no facts; never tell anyone to buy, sell, apply, invest or avoid; never call the IPO good, bad, safe or worth it.\nAnswer with the rewrite only.'
MAX_NEW_TOKENS = 160

In [ ]:
import json, math, random
from pathlib import Path

def read_rows(path):
    with Path(path).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def encode_row(tokenizer, row, max_seq):
    """input_ids + labels with the prompt masked (-100); None when longer than max_seq."""
    user, answer = row["messages"][0], row["messages"][1]["content"]
    prompt = tokenizer.apply_chat_template([user], tokenize=False, add_generation_prompt=True)
    full = tokenizer.apply_chat_template([user, {"role": "assistant", "content": answer}], tokenize=False)
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tokenizer(full, add_special_tokens=False)["input_ids"]
    if len(f_ids) > max_seq or f_ids[: len(p_ids)] != p_ids:
        return None
    return {"input_ids": f_ids, "labels": [-100] * len(p_ids) + f_ids[len(p_ids):]}

def collate(batch, pad_id):
    n = max(len(b["input_ids"]) for b in batch)
    ids = [b["input_ids"] + [pad_id] * (n - len(b["input_ids"])) for b in batch]
    labels = [b["labels"] + [-100] * (n - len(b["labels"])) for b in batch]
    mask = [[1] * len(b["input_ids"]) + [0] * (n - len(b["input_ids"])) for b in batch]
    return ids, labels, mask

In [ ]:
import json, random, shutil, time
from pathlib import Path
import numpy as np
import torch
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer,
                          TrainerCallback, TrainingArguments)

assert PRECISION in ("bf16", "nf4"), PRECISION
JOB_DIR = mount_drive(JOB)
out = Path("/content/student")
out.mkdir(parents=True, exist_ok=True)
ck = Checkpointer(local_dir=out / "ck", drive_dir=JOB_DIR / "ck", every=10**9)  # synced on every save
ckpt_dir = ck.local / "trainer"

train_rows, dev_rows = read_rows(JOB_DIR / "train.jsonl"), read_rows(JOB_DIR / "dev.jsonl")
assert not {r["company"] for r in train_rows} & {r["company"] for r in dev_rows}, "company on both sides"
if SMOKE:
    train_rows, dev_rows = train_rows[:64], dev_rows[:16]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
train = [e for e in (encode_row(tokenizer, r, MAX_SEQ) for r in train_rows) if e]
dev = [e for e in (encode_row(tokenizer, r, MAX_SEQ) for r in dev_rows) if e]
print(len(train), "train |", len(dev), "dev | skipped as too long:", len(train_rows) + len(dev_rows) - len(train) - len(dev))

class Rows(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collator(batch):
    ids, labels, mask = collate(batch, pad_id)
    return {"input_ids": torch.tensor(ids), "labels": torch.tensor(labels), "attention_mask": torch.tensor(mask)}

if PRECISION == "nf4":  # the T4 recipe: 4-bit base, fp16 compute
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, torch_dtype=torch.float16,
                                                 device_map="auto")
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:  # L4/A100: LoRA on a bf16 base, no quantisation
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

class SyncToDrive(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        ck.sync()  # a disconnect loses at most SAVE_STEPS steps

args = TrainingArguments(
    output_dir=str(ckpt_dir), num_train_epochs=EPOCHS, max_steps=10 if SMOKE else -1,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LEARNING_RATE, lr_scheduler_type="cosine",
    warmup_ratio=0.03, fp16=(PRECISION == "nf4"), bf16=(PRECISION == "bf16"), gradient_checkpointing=True,
    logging_steps=10, eval_strategy="steps", eval_steps=SAVE_STEPS, save_strategy="steps",
    save_steps=SAVE_STEPS, save_total_limit=2, report_to="none", seed=SEED, remove_unused_columns=False)
trainer = Trainer(model=model, args=args, train_dataset=Rows(train), eval_dataset=Rows(dev),
                  data_collator=collator, callbacks=[SyncToDrive()])
t0 = time.time()
resume = ck.latest_trainer_checkpoint("trainer")
print("resuming from", resume) if resume else print("fresh run")
trainer.train(resume_from_checkpoint=resume)
eval_loss = trainer.evaluate()["eval_loss"]
adapter = out / "adapter"
model.save_pretrained(str(adapter)); tokenizer.save_pretrained(str(adapter))
metrics = {"model": MODEL, "precision": PRECISION, "epochs": EPOCHS, "smoke": SMOKE, "n_train": len(train),
           "n_dev": len(dev), "dev_loss": eval_loss, "train_runtime_s": round(time.time() - t0),
           "log_history": trainer.state.log_history, "gpu": torch.cuda.get_device_name(0)}
del model, trainer; torch.cuda.empty_cache()

# ---- merge (temporary, only to make the GGUF; the merged fp16 weights are deleted below) --------
gguf = out / "simplifier-q4_k_m.gguf"
if EXPORT_GGUF:
    base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cpu")
    merged = PeftModel.from_pretrained(base, str(adapter)).merge_and_unload()
    merged.save_pretrained(str(out / "merged"), safe_serialization=True); tokenizer.save_pretrained(str(out / "merged"))
    del base, merged
    run = lambda cmd: subprocess.run(cmd, check=True, shell=True)
    lc = Path("/tmp/llama.cpp")
    if not lc.exists():
        run(f"git clone --depth 1 https://github.com/ggml-org/llama.cpp {lc}")
    run(f"{sys.executable} -m pip install -q -r {lc}/requirements/requirements-convert_hf_to_gguf.txt")
    run(f"{sys.executable} {lc}/convert_hf_to_gguf.py {out / 'merged'} --outtype f16 --outfile {out / 'f16.gguf'}")
    run(f"cmake -S {lc} -B {lc}/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF && cmake --build {lc}/build --target llama-quantize -j 4")
    run(f"{lc}/build/bin/llama-quantize {out / 'f16.gguf'} {gguf} Q4_K_M")
    (out / "f16.gguf").unlink()
    shutil.rmtree(out / "merged")  # never kept: merged fp16 weights do not go to Drive or the Hub
    metrics["gguf_mb"] = round(gguf.stat().st_size / 2**20)
(out / "metrics.json").write_text(json.dumps(metrics, indent=1), encoding="utf-8")

# ---- keep the small files on Drive; optionally upload to a private HF repo --------------------------
keep = JOB_DIR / "out"
keep.mkdir(parents=True, exist_ok=True)
shutil.copytree(adapter, keep / "adapter", dirs_exist_ok=True)
shutil.copy(out / "metrics.json", keep / "metrics.json")
if EXPORT_GGUF:
    shutil.copy(gguf, keep / gguf.name)
if UPLOAD:
    from google.colab import userdata
    from huggingface_hub import HfApi
    api = HfApi(token=userdata.get("HF_TOKEN"))
    api.create_repo(HF_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=str(adapter), path_in_repo="adapter", repo_id=HF_REPO)
    if EXPORT_GGUF:
        api.upload_file(path_or_fileobj=str(gguf), path_in_repo=gguf.name, repo_id=HF_REPO)
    print("uploaded adapter + GGUF to the private repo", HF_REPO)
print("STUDENT OK", json.dumps({k: v for k, v in metrics.items() if k != "log_history"}))

In [ ]:
s = run_summary(JOB_DIR, "student", started=STARTED, units_before=UNITS_BEFORE, units_after=UNITS_AFTER,
                items_done=len(train), pins={"torch": torch.__version__}, extra={"precision": PRECISION, "smoke": SMOKE})
print(json.dumps(s, indent=1))
unassign_runtime(UNASSIGN)